In [30]:
!pip install torch torchvision torchaudio --index-url  https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [31]:
!pip install numpy pandas matplotlib seaborn scikit-learn tqdm


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [32]:
!nvidia-smi

Fri Mar 20 09:51:20 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce GTX 1660        Off | 00000000:01:00.0  On |                  N/A |
| 39%   38C    P8              12W / 120W |    489MiB /  6144MiB |     14%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# CSV 파일 불러오기
df = pd.read_csv('train.csv')
print("데이터 로드 완료!")


데이터 로드 완료!


In [34]:
print("데이터 크기(행, 열):", df.shape)
display(df.head())
print("[결측치 개수 확인]")
print(df.isnull().sum())
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
print("불필요한 열 제거 완료. 현재 열 목록:", df.columns.tolist())
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Embarked'] = df['Embarked'].fillna('S')

# 결측치가 잘 채워졌는지 재확인
print("남은 결측치 개수:\n", df.isnull().sum())

데이터 크기(행, 열): (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


[결측치 개수 확인]
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
불필요한 열 제거 완료. 현재 열 목록: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
남은 결측치 개수:
 Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [35]:
# 성별(Sex) 이진 매핑: 남성은 0, 여성은 1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# 탑승항구(Embarked) 원-핫 인코딩: 3개의 카테고리(S, C, Q)를 별도의 열(0과 1)로 분리
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

display(df.head())


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,False,True
1,1,1,1,38.0,1,0,71.2833,False,False
2,1,3,1,26.0,0,0,7.9250,False,True
3,1,1,1,35.0,1,0,53.1000,False,True
4,0,3,0,35.0,0,0,8.0500,False,True


In [36]:
from sklearn.model_selection import train_test_split

# 특성(X)과 정답(y) 분리
X = df.drop('Survived', axis=1).values
y = df['Survived'].values

# 학습/검증 데이터 분리 (Random state를 고정하여 항상 동일하게 분리되도록 함)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"학습용 데이터 크기: {X_train.shape}")
print(f"검증용 데이터 크기: {X_val.shape}")




학습용 데이터 크기: (712, 8)
검증용 데이터 크기: (179, 8)


In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 학습 데이터(X_train)의 평균과 분산을 계산(fit)하고 적용(transform)
X_train = scaler.fit_transform(X_train)

# 검증 데이터(X_val)는 학습 데이터의 기준을 그대로 적용(transform만 수행)
X_val = scaler.transform(X_val)

print("스케일링 적용 후 데이터 예시 (X_train 첫 번째 행):")
print(X_train[0])


스케일링 적용 후 데이터 예시 (X_train 첫 번째 행):
[-1.61413602 -0.7243102   1.22920747 -0.47072241 -0.47934164 -0.07868358
 -0.30335547  0.59248936]


In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

class TitanicDataset(Dataset):
    def __init__(self, X, y):
        # Numpy 배열을 PyTorch Tensor(float32)로 변환
        self.X = torch.tensor(X, dtype=torch.float32)
        # 정답(y)의 차원을 손실 함수 계산에 맞게 (N, 1) 행렬 형태로 재구조화
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 인스턴스 생성
train_dataset = TitanicDataset(X_train, y_train)
val_dataset = TitanicDataset(X_val, y_val)
print("Dataset 객체 생성 완료!")


Dataset 객체 생성 완료!


In [39]:
# 학습 데이터는 순서를 무작위로 섞어(shuffle=True) 과적합을 방지합니다.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 첫 번째 배치의 데이터 형태 확인
for inputs, labels in train_loader:
    print("입력 텐서 형태 (배치크기, 특성수):", inputs.shape)
    print("정답 텐서 형태 (배치크기, 1):", labels.shape)
    break # 하나만 확인하고 반복문 종료


입력 텐서 형태 (배치크기, 특성수): torch.Size([32, 8])
정답 텐서 형태 (배치크기, 1): torch.Size([32, 1])


In [40]:
import torch.nn as nn

class TitanicModel(nn.Module):
    def __init__(self, input_size):
        super(TitanicModel, self).__init__()
        # 입력층 -> 은닉층 1: 입력된 특성을 64차원 벡터로 투영
        self.layer1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU() # 비선형 활성화 함수 (ReLU)

        # 은닉층 1 -> 은닉층 2: 64차원을 32차원으로 압축
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()

        # 은닉층 2 -> 출력층: 최종적으로 1차원(스칼라) 값으로 출력
        self.output_layer = nn.Linear(32, 1)
        # 이진 분류를 위해 출력값을 0~1 사이의 확률값으로 매핑하는 Sigmoid 함수
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 데이터가 신경망을 통과하는 순전파(Forward Propagation) 연산 순서
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.sigmoid(self.output_layer(x))
        return x

# 모델 생성 (입력 차원은 전처리된 특성의 개수인 8)
model = TitanicModel(input_size=X_train.shape[1])
print(model)


TitanicModel(
  (layer1): Linear(in_features=8, out_features=64, bias=True)
  (relu1): ReLU()
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [41]:
import torch.optim as optim

# 손실 함수 (이진 분류용 Binary Cross Entropy Loss)
criterion = nn.BCELoss()
# 최적화 알고리즘 (학습률 0.001의 Adam Optimizer)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50 # 전체 데이터를 50번 반복 학습

for epoch in range(epochs):
    # 1. 학습(Train) 페이즈
    model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        optimizer.zero_grad()            # 기울기 초기화
        outputs = model(inputs)          # 순전파 (예측)
        loss = criterion(outputs, labels)# 오차 계산
        loss.backward()                  # 역전파 (기울기 계산)
        optimizer.step()                 # 파라미터 업데이트

        train_loss += loss.item()

    # 2. 검증(Validation) 페이즈
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad(): # 검증 시에는 역전파를 위한 기울기 연산을 중지
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            # 예측 확률이 0.5 이상이면 1(생존), 미만이면 0(사망)으로 매핑
            predicted = (outputs >= 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # 에폭 10번마다 결과 출력
    if (epoch + 1) % 10 == 0:
        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {train_loss/len(train_loader):.4f} | "
              f"Val Accuracy: {accuracy:.2f}%")

print("🎉 모델 학습이 완료되었습니다.")


Epoch [10/50] | Train Loss: 0.4041 | Val Accuracy: 80.45%
Epoch [20/50] | Train Loss: 0.3787 | Val Accuracy: 81.56%
Epoch [30/50] | Train Loss: 0.3722 | Val Accuracy: 81.56%
Epoch [40/50] | Train Loss: 0.3548 | Val Accuracy: 81.01%
Epoch [50/50] | Train Loss: 0.3510 | Val Accuracy: 79.89%
🎉 모델 학습이 완료되었습니다.


In [42]:
# 테스트 데이터 로드
test_df = pd.read_csv('test.csv')

# 제출 파일 양식을 위해 승객 ID 분리
passenger_ids = test_df['PassengerId']

# 불필요한 열 제거
test_df = test_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# 결측치 대체 (test.csv에는 Fare 결측치도 1개 존재합니다)
test_df['Age'] = test_df['Age'].fillna(test_df['Age'].mean())
test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].mean())
test_df['Embarked'] = test_df['Embarked'].fillna('S')

# 인코딩
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})
test_df = pd.get_dummies(test_df, columns=['Embarked'], drop_first=True)

# 피처 스케일링 (중요: 학습 데이터의 기준을 유지하기 위해 transform만 호출)
X_test = scaler.transform(test_df.values)

# 텐서 변환
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
print("테스트 데이터 전처리 및 텐서 변환 완료!")


테스트 데이터 전처리 및 텐서 변환 완료!


In [43]:
model.eval() # 추론 모드
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    # 확률을 0과 1의 클래스로 이진화하고 1차원 정수 배열로 변환
    predictions = (test_outputs >= 0.5).int().numpy().flatten()

# DataFrame 생성
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions
})

# 결과를 CSV 파일로 추출
submission.to_csv('submission.csv', index=False)
print("예측 결과 파일(submission.csv)이 성공적으로 생성되었습니다!")
display(submission.head())


예측 결과 파일(submission.csv)이 성공적으로 생성되었습니다!


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,0
